In [ ]:
# Import

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

# Matplotlib beállítások
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

print("=" * 80)
print("TIPFORGE PÉNZÜGYI SZIMULÁCIÓ - MONTE CARLO MODELL")
print("=" * 80)

In [ ]:
# ALAPPARAMETEREK DEFINIÁLÁSA

# Árazási szcenáriók
pricing_scenarios = {
    'Aggressive_Low': 2990,
    'Survey_Median': 5000,
    'Project_Plan': 7900,
    'Market_Standard': 12000
}

# Marketing költségek (Ft/hó)
marketing_budget_scenarios = {
    'Minimal': 20000,
    'Moderate': 50000,
    'Aggressive': 100000
}

# Elérési becslések (reach per 10k Ft marketing)
reach_per_10k = {
    'Facebook_Ads': 2000,  # impressziók
    'Google_Ads': 1500,
    'Instagram': 2500,
    'Organic': 500
}

# Konverziós tölcsér
funnel_rates = {
    'impression_to_click': 0.02,      # 2% CTR
    'click_to_landing': 0.80,         # 80% eljut landing oldalra
    'landing_to_signup': 0.10,        # 10% feliratkozik (email)
    'signup_to_trial': 0.25,          # 25% kipróbálja (profitgaranciával)
    'trial_to_paid': 0.30             # 60% fizető lesz (ha nyereséges volt)
}

# Churn rate (havi lemorzsolódás)
monthly_churn_base = 0.40  # 15% havonta

# Profitgarancia kifizetés
profit_guarantee_trigger = 0.30  # 30% új ügyfél veszteséges első hónapban

# Fix költségek (Ft/hó)
fixed_costs = {
    'hosting_domain': 0,
    'accounting': 0,
    'payment_gateway': 0,
    'tools_software': 0,
    'customer_support': 0
}

# Változó költségek
variable_cost_per_customer = 500  # Ft/ügyfél/hó
payment_fee_rate = 0.025  # 2.5% tranzakciós díj

In [ ]:
# MONTE CARLO SZIMULÁCIÓ - 12 HÓNAP

def simulate_month(month, current_customers, price, marketing_budget, 
                   organic_signups=0, profit_guarantee_payout_rate=0.3):
    """
    Egy hónap szimulálása
    """
    # Marketing reach
    total_reach = (marketing_budget / 10000) * np.random.normal(2000, 300)
    
    # Organikus reach
    organic_reach = organic_signups * 50  # minden organikus signup hoz 50 impressziót
    
    total_impressions = total_reach + organic_reach
    
    # Konverziós tölcsér
    clicks = int(total_impressions * np.random.normal(
        funnel_rates['impression_to_click'], 0.005))
    
    landing_visits = int(clicks * funnel_rates['click_to_landing'])
    
    email_signups = int(landing_visits * np.random.normal(
        funnel_rates['landing_to_signup'], 0.02))
    
    trial_users = int(email_signups * funnel_rates['signup_to_trial']) + organic_signups
    
    # Profitgarancia aktiválódik?
    guarantee_triggered = int(trial_users * profit_guarantee_payout_rate)
    paid_conversions = int((trial_users - guarantee_triggered) * 
                           np.random.normal(funnel_rates['trial_to_paid'], 0.1))
    
    # Új fizetős ügyfelek
    new_paying_customers = max(0, paid_conversions)
    
    # Churn (lemorzsolódás)
    # Churn függ az ártól és az időtől
    price_churn_multiplier = 1 + ((price - 5000) / 10000) * 0.5  # drágább = nagyobb churn
    month_churn_multiplier = 1 - (month * 0.02)  # idővel javul a megtartás
    
    actual_churn_rate = monthly_churn_base * price_churn_multiplier * month_churn_multiplier
    actual_churn_rate = max(0.05, min(0.30, actual_churn_rate))  # 5-30% között
    
    churned_customers = int(current_customers * actual_churn_rate)
    
    # Hónap végi ügyfélszám
    end_customers = current_customers + new_paying_customers - churned_customers
    
    # Bevételek
    subscription_revenue = current_customers * price
    
    # Költségek
    total_fixed_costs = sum(fixed_costs.values())
    variable_costs = current_customers * variable_cost_per_customer
    payment_fees = subscription_revenue * payment_fee_rate
    
    # Profitgarancia visszafizetés
    # Azok akik az első hónapban vannak és veszteségesek voltak
    guarantee_payout = guarantee_triggered * price
    
    # Teljes költség
    total_costs = (total_fixed_costs + variable_costs + payment_fees + 
                   marketing_budget + guarantee_payout)
    
    # Nettó profit
    net_profit = subscription_revenue - total_costs
    
    return {
        'month': month,
        'impressions': int(total_impressions),
        'clicks': clicks,
        'email_signups': email_signups,
        'trial_users': trial_users,
        'new_paying': new_paying_customers,
        'churned': churned_customers,
        'total_customers': end_customers,
        'churn_rate': actual_churn_rate,
        'revenue': subscription_revenue,
        'costs': total_costs,
        'marketing_cost': marketing_budget,
        'guarantee_payout': guarantee_payout,
        'net_profit': net_profit,
        'cumulative_profit': 0  # később frissítjük
    }

def run_simulation(price, marketing_budget, months=12, organic_per_month=2):
    """
    12 hónapos szimuláció futtatása
    """
    results = []
    current_customers = 0
    cumulative_profit = 0
    
    for month in range(1, months + 1):
        month_result = simulate_month(
            month, 
            current_customers, 
            price, 
            marketing_budget,
            organic_signups=organic_per_month
        )
        
        cumulative_profit += month_result['net_profit']
        month_result['cumulative_profit'] = cumulative_profit
        
        current_customers = month_result['total_customers']
        results.append(month_result)
    
    return pd.DataFrame(results)

In [ ]:
# SZCENÁRIÓK FUTTATÁSA

print("\n" + "=" * 80)
print("SZCENÁRIÓK FUTTATÁSA")
print("=" * 80)

scenarios = []

for price_name, price in pricing_scenarios.items():
    for marketing_name, marketing in marketing_budget_scenarios.items():
        scenario_name = f"{price_name}_{marketing_name}"
        
        # Futtatunk N szimulációt és átlagoljuk
        n_simulations = 100
        all_simulations = []
        
        for i in range(n_simulations):
            sim_result = run_simulation(price, marketing, months=12, organic_per_month=2)
            sim_result['simulation'] = i
            sim_result['scenario'] = scenario_name
            sim_result['price'] = price
            sim_result['marketing_budget'] = marketing
            all_simulations.append(sim_result)
        
        combined = pd.concat(all_simulations)
        
        # Aggregált statisztikák
        final_month = combined[combined['month'] == 12]
        
        scenarios.append({
            'scenario': scenario_name,
            'price': price,
            'marketing': marketing,
            'avg_final_customers': final_month['total_customers'].mean(),
            'avg_final_profit': final_month['cumulative_profit'].mean(),
            'median_final_profit': final_month['cumulative_profit'].median(),
            'profit_std': final_month['cumulative_profit'].std(),
            'probability_profitable': (final_month['cumulative_profit'] > 0).sum() / len(final_month) * 100,
            'avg_monthly_revenue_12': final_month['revenue'].mean(),
            'total_guarantee_payout': combined.groupby('simulation')['guarantee_payout'].sum().mean()
        })

scenarios_df = pd.DataFrame(scenarios)

print("\nTop 5 szcenárió (kumulatív profit alapján):")
print(scenarios_df.nlargest(5, 'avg_final_profit')[
    ['scenario', 'price', 'marketing', 'avg_final_customers', 'avg_final_profit', 'probability_profitable']
])

In [ ]:
# VIZUALIZÁCIÓK

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('TipForge - Monte Carlo Pénzügyi Szimuláció (12 hónap)', 
             fontsize=16, fontweight='bold')

# 4.1 Végső profit vs Ár (marketing szintenként)
for marketing_name, marketing in marketing_budget_scenarios.items():
    subset = scenarios_df[scenarios_df['marketing'] == marketing]
    axes[0, 0].plot(subset['price'], subset['avg_final_profit'], 
                    'o-', linewidth=2, markersize=10, label=f'{marketing_name} ({marketing/1000:.0f}k Ft/hó)')

axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)
axes[0, 0].set_xlabel('Ár (Ft/hó)')
axes[0, 0].set_ylabel('Kumulatív profit 12 hónap után (Ft)')
axes[0, 0].set_title('Profitabilitás különböző ár-marketing kombinációkban', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 4.2 Profitabilitás valószínűsége
pivot_prob = scenarios_df.pivot_table(
    values='probability_profitable', 
    index='price', 
    columns='marketing'
)

im = axes[0, 1].imshow(pivot_prob.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)
axes[0, 1].set_xticks(range(len(pivot_prob.columns)))
axes[0, 1].set_xticklabels([f'{m/1000:.0f}k' for m in pivot_prob.columns])
axes[0, 1].set_yticks(range(len(pivot_prob.index)))
axes[0, 1].set_yticklabels([f'{p/1000:.0f}k' for p in pivot_prob.index])
axes[0, 1].set_xlabel('Marketing költség (Ft/hó)')
axes[0, 1].set_ylabel('Ár (Ft/hó)')
axes[0, 1].set_title('Profitabilitás valószínűsége (%)', fontweight='bold')

# Értékek kiírása
for i in range(len(pivot_prob.index)):
    for j in range(len(pivot_prob.columns)):
        text = axes[0, 1].text(j, i, f'{pivot_prob.values[i, j]:.0f}%',
                              ha="center", va="center", color="black", fontsize=9)

cbar = plt.colorbar(im, ax=axes[0, 1])
cbar.set_label('Profitabilitás (%)')

# 4.3 Ügyfélszám növekedés (legjobb 3 szcenárió)
top_3_scenarios = scenarios_df.nlargest(3, 'avg_final_profit')

# Futtassuk újra a top 3-at részletes adatokkal
for idx, row in top_3_scenarios.iterrows():
    sim_data = run_simulation(row['price'], row['marketing'], months=12)
    axes[1, 0].plot(sim_data['month'], sim_data['total_customers'], 
                   'o-', linewidth=2, markersize=8, 
                   label=f"{row['scenario']}\n({row['price']:.0f} Ft, {row['marketing']/1000:.0f}k marketing)")

axes[1, 0].set_xlabel('Hónap')
axes[1, 0].set_ylabel('Összes aktív ügyfél')
axes[1, 0].set_title('Ügyfélszám növekedés (Top 3 szcenárió)', fontweight='bold')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# 4.4 Havi cash flow (legjobb szcenárió)
best_scenario = scenarios_df.nlargest(1, 'avg_final_profit').iloc[0]
best_sim = run_simulation(best_scenario['price'], best_scenario['marketing'], months=12)

axes[1, 1].bar(best_sim['month'] - 0.2, best_sim['revenue'], 
              width=0.4, label='Bevétel', color='green', alpha=0.7)
axes[1, 1].bar(best_sim['month'] + 0.2, best_sim['costs'], 
              width=0.4, label='Költségek', color='red', alpha=0.7)
axes[1, 1].plot(best_sim['month'], best_sim['net_profit'], 
               'o-', linewidth=2, markersize=8, color='blue', label='Nettó profit')
axes[1, 1].axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)

axes[1, 1].set_xlabel('Hónap')
axes[1, 1].set_ylabel('Összeg (Ft)')
axes[1, 1].set_title(f'Havi cash flow - Legjobb szcenárió\n{best_scenario["scenario"]}', 
                    fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('11_monte_carlo_simulation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# RÉSZLETES BREAKDOWN - LEGJOBB SZCENÁRIÓ

print("\n" + "=" * 80)
print("LEGJOBB SZCENÁRIÓ RÉSZLETES ELEMZÉSE")
print("=" * 80)

print(f"\nSzcenárió: {best_scenario['scenario']}")
print(f"Ár: {best_scenario['price']:,.0f} Ft/hó")
print(f"Marketing költség: {best_scenario['marketing']:,.0f} Ft/hó")
print(f"\n12 hónapos eredmények:")
print(f"  • Átlagos végső ügyfélszám: {best_scenario['avg_final_customers']:.0f} fő")
print(f"  • Kumulatív profit: {best_scenario['avg_final_profit']:,.0f} Ft")
print(f"  • Profitabilitás valószínűsége: {best_scenario['probability_profitable']:.1f}%")
print(f"  • 12. havi bevétel: {best_scenario['avg_monthly_revenue_12']:,.0f} Ft")
print(f"  • Összes profitgarancia kifizetés: {best_scenario['total_guarantee_payout']:,.0f} Ft")

print("\n" + "-" * 80)
print("Hónap-hónap részletezés (legjobb szcenárió átlaga):")
print("-" * 80)

for idx, row in best_sim.iterrows():
    print(f"\n{row['month']:2.0f}. hónap:")
    print(f"  Új ügyfelek: {row['new_paying']:3.0f} | Lemorzsolódás: {row['churned']:3.0f} | Összes: {row['total_customers']:4.0f}")
    print(f"  Bevétel: {row['revenue']:>10,.0f} Ft | Költség: {row['costs']:>10,.0f} Ft | Profit: {row['net_profit']:>10,.0f} Ft")
    print(f"  Kumulatív profit: {row['cumulative_profit']:>10,.0f} Ft")

In [ ]:
# ÉRZÉKENYSÉGVIZSGÁLAT

print("\n" + "=" * 80)
print("ÉRZÉKENYSÉGVIZSGÁLAT - KULCSVÁLTOZÓK HATÁSA")
print("=" * 80)

# Konverziós ráta érzékenység
conversion_rates = [0.05, 0.10, 0.15, 0.20]
conversion_results = []

base_price = 5000
base_marketing = 50000

for conv_rate in conversion_rates:
    # Módosítjuk a globális funnel_rates-t
    original_rate = funnel_rates['landing_to_signup']
    funnel_rates['landing_to_signup'] = conv_rate
    
    sim = run_simulation(base_price, base_marketing, months=12)
    final_profit = sim.iloc[-1]['cumulative_profit']
    final_customers = sim.iloc[-1]['total_customers']
    
    conversion_results.append({
        'conversion_rate': conv_rate * 100,
        'final_profit': final_profit,
        'final_customers': final_customers
    })
    
    # Visszaállítjuk
    funnel_rates['landing_to_signup'] = original_rate

conversion_df = pd.DataFrame(conversion_results)

print("\nKonverziós ráta hatása (landing → signup):")
print(conversion_df.to_string(index=False))

# Churn rate érzékenység
churn_rates = [0.05, 0.10, 0.15, 0.20, 0.25]
churn_results = []

for churn_rate in churn_rates:
    original_churn = monthly_churn_base
    monthly_churn_base = churn_rate
    
    sim = run_simulation(base_price, base_marketing, months=12)
    final_profit = sim.iloc[-1]['cumulative_profit']
    final_customers = sim.iloc[-1]['total_customers']
    
    churn_results.append({
        'churn_rate': churn_rate * 100,
        'final_profit': final_profit,
        'final_customers': final_customers
    })
    
    monthly_churn_base = original_churn

churn_df = pd.DataFrame(churn_results)

print("\nChurn rate hatása:")
print(churn_df.to_string(index=False))

In [ ]:
# ÖSSZEFOGLALÓ AJÁNLÁSOK

print("\n" + "=" * 80)
print("STRATÉGIAI AJÁNLÁSOK A SZIMULÁCIÓ ALAPJÁN")
print("=" * 80)

print("\n🎯 OPTIMÁLIS STRATÉGIA:")
print(f"  • Javasolt ár: {best_scenario['price']:,.0f} Ft/hó")
print(f"  • Javasolt marketing költség: {best_scenario['marketing']:,.0f} Ft/hó")
print(f"  • Várható 12 hónapos profit: {best_scenario['avg_final_profit']:,.0f} Ft")
print(f"  • Sikeres break-even valószínűség: {best_scenario['probability_profitable']:.0f}%")

print("\n⚠️  KRITIKUS KOCKÁZATOK:")
print("  1. Alacsony konverziós ráta (landing → signup): 10% még optimista lehet")
print("  2. Magas churn rate: 15% havonta agresszív megtartási stratégiát igényel")
print("  3. Profitgarancia: 30% új ügyfél veszteséges = jelentős cash flow kockázat")
print("  4. Organikus növekedés: 2 fő/hó nagyon alacsony, közösségi jelenlét kulcsfontosságú")

print("\n💡 ALTERNATÍV SZCENÁRIÓK:")
top_5 = scenarios_df.nlargest(5, 'avg_final_profit')
for idx, (i, row) in enumerate(top_5.iterrows(), 1):
    print(f"  {idx}. {row['scenario']}: {row['avg_final_profit']:,.0f} Ft profit, "
          f"{row['probability_profitable']:.0f}% siker valószínűség")

print("\n" + "=" * 80)
print("SZIMULÁCIÓ BEFEJEZVE!")
print("Mentett kép: 11_monte_carlo_simulation.png")
print("=" * 80)